In [0]:
%run "./utils/ingest_raw_to_bronze_utils"

In [0]:
from pyspark.sql.functions import *

catalog = "novamart"
schema = "bronze"
check_point = '/Volumes/novamart/bronze/checkpoint/'

# -----------------------------------
#  Sql table ingestion
# -----------------------------------

tables = {
    "sql_customers" : "customers",
    "sql_products" : "products",
    "sql_sales_transactions" : "sales_transactions"
}

base_path = f"/Volumes/{catalog}/{schema}/landing/landing_zone/sales_db/"

for table_name, folder in tables.items():

    ingest_raw_to_bronze(
        source_path = f"{base_path}/{folder}",
        target_table = f"{catalog}.{schema}.{table_name}",
        checkpoint_path = f"{check_point}/{table_name}_bronze",
        file_format = "csv",
        extra_read_options= {"header": "true", "cloudFiles.inferSchema": "true"}
    )

# ------------------------------------------------------
#  Crm customers ingestion (json format with multiline)
# ------------------------------------------------------

ingest_raw_to_bronze(
        source_path = f"/Volumes/{catalog}/{schema}/landing/landing_zone/crm_api/customers",
        target_table= f"{catalog}.{schema}.crm_customers",
        checkpoint_path = f"{check_point}/crm_customers_bronze",
        file_format = "json",
        extra_read_options= {"cloudFiles.inferColumnsTypes": "true", "multiline" : "true"}
)

# ------------------------------------------------------
#  Kafka clickstream ingestion (jsonl format)
# ------------------------------------------------------
ingest_raw_to_bronze(
        source_path = f"/Volumes/{catalog}/{schema}/landing/landing_zone/kafka",
        target_table= f"{catalog}.{schema}.clickstream",
        checkpoint_path = f"/Volumes/{catalog}/{schema}/checkpoint/kafka_clcikstream_bronze",
        file_format = "json",
        extra_read_options= {"cloudFiles.inferColumnsTypes": "true"}
)

In [0]:
catalog = 'novamart'
schema = 'bronze'
df = spark.read.table(f"{catalog}.{schema}.sql_customers")
display(df.limit(50))

In [0]:
catalog = 'novamart'
schema = 'bronze'
df = spark.read.table(f"{catalog}.{schema}.crm_customers")
display(df.limit(50))

In [0]:
catalog = 'novamart'
schema = 'bronze'
df = spark.read.table(f"{catalog}.{schema}.sql_products")
display(df.limit(50))

In [0]:
catalog = 'novamart'
schema = 'bronze'
df = spark.read.table(f"{catalog}.{schema}.sql_sales_transactions")
display(df.limit(50))

In [0]:
catalog = 'novamart'
schema = 'bronze'
df = spark.read.table(f"{catalog}.{schema}.clickstream")
display(df.limit(50))